# Absent target: NK signature on blood with NK cells removed (reviewer #2, major concern)

Domínguez Conde et al. blood (BLD), SIGMA input file. One NK configuration, fixed before
looking at the results, run twice with `find_target.py` (no QC, no harmonization, no merge):

- **all NK**: every cell (positive control, compared with the author labels)
- **no NK**: every NK-labelled cell removed (the reviewer's test)

For each run: candidates, whether a target component passed M, cells with score ≥ 0.95 and
their author labels.

In [1]:
import json
import os
import re
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

In [2]:
INPUT = ("/mnt/projects_tn03/AD_EB_singlecell/articleSIGMA_results-2/Dominguez-Data/forSIGMA/"
         "Dominguez_science_science-abl5197_BLD_data.h5ad")
SIGMA_SCRIPTS = "/home/lemgui01/SIGMA/scripts"
WORK_DIR = "results/nk_absent"
FIG_DIR = "figures/nk_absent"

# Author label column: the first one found is used
LABEL_COLUMNS = ["Manually_curated_celltype", "cell_type", "celltype", "Majority_voting_CellTypist"]
NK_PATTERN = r"^NK"  # author labels counted as NK; check the printed list below

# NK configuration, fixed before the runs
CANDIDATE = ["NKG7", "GNLY", "KLRD1", "PRF1"]
TARGET = ["KLRF1", "NCAM1", "SH2D1B", "NCR1", "FCGR3A", "KLRC1", "GNLY", "NKG7", "PRF1", "GZMB"]
EXCLUDE = {
    "T_cells": ["CD3E", "CD3D", "CD3G", "TRAC", "TRBC1", "TRBC2", "CD5", "CD28"],
    "B_cells": ["MS4A1", "CD79A", "CD79B", "CD19", "PAX5"],
    "Monocytes": ["CD14", "LYZ", "FCN1", "CST3", "S100A8", "S100A9"],
}
EXCLUDE_CELLTYPES = "True"  # exclusion signatures are full competing lineages
MIN_GENES_DETECTED = 2
GENE_DETECTION_THRESHOLD = 2
MIN_MEAN_EXPRESSION = 2
SCORE_THRESHOLD = 0.95

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Data and NK labels

In [3]:
adata = sc.read_h5ad(INPUT)
label_col = next(c for c in LABEL_COLUMNS if c in adata.obs.columns)
labels = adata.obs[label_col].astype(str)
is_nk = labels.str.contains(NK_PATTERN, regex=True).values
print(f"{adata.n_obs:,} cells, label column: {label_col}")
print("NK labels:", sorted(labels[is_nk].unique()), f"({is_nk.sum():,} cells)")
missing = [g for g in CANDIDATE + TARGET + sum(EXCLUDE.values(), []) if g not in adata.var_names]
print("Genes absent from var_names (SIGMA maps aliases through BioMart):", missing)
labels.value_counts().head(30)

27,620 cells, label column: Manually_curated_celltype
NK labels: ['NK_CD16+', 'NK_CD56bright_CD16-'] (3,984 cells)
Genes absent from var_names (SIGMA maps aliases through BioMart): []


Manually_curated_celltype
Tnaive/CM_CD4             6609
Classical monocytes       5850
Teffector/EM_CD4          4331
NK_CD16+                  3539
Tem/emra_CD8              1983
Tnaive/CM_CD8             1182
Tregs                      798
Trm/em_CD8                 635
NK_CD56bright_CD16-        445
T_CD4/CD8                  270
Nonclassical monocytes     236
MNP/T doublets             229
Cycling T&NK               215
Naive B cells              194
Memory B cells             171
Megakaryocytes             163
Tfh                        111
MAIT                       107
DC2                        100
Plasma cells                66
T/B doublets                62
Progenitor                  61
Tgd_CRTAM+                  47
pDC                         43
Trm_Th1/Th17                35
Plasmablasts                35
Mast cells                  22
ABCs                        21
ILC3                        15
DC1                         14
Name: count, dtype: int64

## 2. Runs

In [4]:
def run_sigma(name, keep):
    # Runs find_target.py on the cells in `keep` and reuses the output when it exists.
    # Absolute paths: find_target.py runs from the SIGMA folder
    work = os.path.abspath(WORK_DIR)
    out = os.path.join(work, f"{name}.h5ad")
    if not os.path.exists(out):
        inp = os.path.join(work, f"input_{name}.h5ad")
        adata[keep].copy().write_h5ad(inp)
        cmd = [sys.executable, os.path.join(SIGMA_SCRIPTS, "find_target.py"),
               "--h5ad_file", inp, "--output_file", out, "--study_name", name,
               "--candidate_genes", *CANDIDATE, "--target_genes", *TARGET,
               "--exclude_genes", json.dumps(EXCLUDE),
               "--min_genes_detected", str(MIN_GENES_DETECTED),
               "--gene_detection_threshold", str(GENE_DETECTION_THRESHOLD),
               "--n_components_target", "kde", "--n_components_exclu", "kde",
               "--min_mean_expression", str(MIN_MEAN_EXPRESSION),
               "--do_QC", "False", "--species", "hsapiens",
               "--plot_folder", os.path.join(work, "plots", name),
               "--exclude_celltypes", EXCLUDE_CELLTYPES]
        with open(os.path.join(work, f"{name}.log"), "w") as log:
            subprocess.run(cmd, check=True, stdout=log, stderr=subprocess.STDOUT,
                           cwd=os.path.dirname(SIGMA_SCRIPTS))
        os.remove(inp)
    return sc.read_h5ad(out).obs


runs = []
for name, keep in [("all_NK", np.ones(adata.n_obs, dtype=bool)), ("no_NK", ~is_nk)]:
    obs = run_sigma(name, keep)
    runs.append({"name": name, "nk_in": int((keep & is_nk).sum()), "cells_in": int(keep.sum()),
                 "obs": obs})
    print(name, f"cells in: {keep.sum():,}", f"NK in: {(keep & is_nk).sum():,}")

CalledProcessError: Command '['/home/lemgui01/.miniconda3/envs/SIGMA/bin/python', '/home/lemgui01/SIGMA/scripts/find_target.py', '--h5ad_file', '/home/lemgui01/SIGMA/notebooks/results/nk_absent/input_all_NK.h5ad', '--output_file', '/home/lemgui01/SIGMA/notebooks/results/nk_absent/all_NK.h5ad', '--study_name', 'all_NK', '--candidate_genes', 'NKG7', 'GNLY', 'KLRD1', 'PRF1', '--target_genes', 'KLRF1', 'NCAM1', 'SH2D1B', 'NCR1', 'FCGR3A', 'KLRC1', 'GNLY', 'NKG7', 'PRF1', 'GZMB', '--exclude_genes', '{"T_cells": ["CD3E", "CD3D", "CD3G", "TRAC", "TRBC1", "TRBC2", "CD5", "CD28"], "B_cells": ["MS4A1", "CD79A", "CD79B", "CD19", "PAX5"], "Monocytes": ["CD14", "LYZ", "FCN1", "CST3", "S100A8", "S100A9"]}', '--min_genes_detected', '2', '--gene_detection_threshold', '2', '--n_components_target', 'kde', '--n_components_exclu', 'kde', '--min_mean_expression', '2', '--do_QC', 'False', '--species', 'hsapiens', '--plot_folder', '/home/lemgui01/SIGMA/notebooks/results/nk_absent/plots/all_NK', '--exclude_ce

## 3. Results

In [ ]:
def read_log(name):
    # Candidate count and M check from the find_target.py log: the output is empty when no
    # target component passes M
    text = open(os.path.join(WORK_DIR, f"{name}.log")).read()
    n = re.search(r"Number of candidate cells: (\d+)", text)
    return int(n.group(1)) if n else 0, "is below threshold" not in text


rows, selected_labels = [], []
for r in runs:
    obs = r["obs"]
    lab = labels.reindex(obs.index).astype(str)
    n_candidates, target_passed_M = read_log(r["name"])
    sel = (obs["score"] >= SCORE_THRESHOLD) if len(obs) and "score" in obs else \
        pd.Series(False, index=obs.index)
    sel_nk = lab[sel].str.contains(NK_PATTERN, regex=True)
    rows.append({"run": r["name"], "cells_in": r["cells_in"], "nk_in": r["nk_in"],
                 "candidates": n_candidates, "target_passed_M": target_passed_M,
                 "selected": int(sel.sum()), "selected_nk": int(sel_nk.sum()),
                 "selected_other": int((~sel_nk).sum()),
                 "recall_nk": sel_nk.sum() / r["nk_in"] if r["nk_in"] else np.nan,
                 "precision_nk": sel_nk.mean() if sel.sum() else np.nan})
    selected_labels.append(lab[sel].value_counts().rename(r["name"]))

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(WORK_DIR, "nk_absent_summary.csv"), index=False)
summary.round(3)

In [ ]:
# Author labels of the selected cells (non-NK labels are the false positives)
labels_table = pd.concat(selected_labels, axis=1).fillna(0).astype(int)
labels_table.to_csv(os.path.join(WORK_DIR, "nk_absent_selected_labels.csv"))
labels_table

In [ ]:
# Author labels of the candidates in the no-NK run: which cells pass the prefilter without NK
no_nk = next(r for r in runs if r["name"] == "no_NK")
labels.reindex(no_nk["obs"].index).value_counts().head(15)

## 4. Figure

In [ ]:
INK, INK2, MUTED, AXIS = "#0b0b0b", "#52514e", "#898781", "#c3c2b7"
BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7, "xtick.labelsize": 6.5,
    "ytick.labelsize": 6.5, "legend.fontsize": 6.5, "axes.edgecolor": AXIS,
    "axes.linewidth": 0.6, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlecolor": INK,
    "pdf.fonttype": 42,
})


def panel_label(ax, letter):
    ax.text(-0.18, 1.04, letter, transform=ax.transAxes, fontsize=10, fontweight="bold",
            color=INK, va="bottom")


fig, axes = plt.subplots(1, 2, figsize=(5.8, 2.4), gridspec_kw={"wspace": 0.45})

# A: score distributions of the candidates
ax = axes[0]
bins = np.linspace(0, 1, 41)
for r, color, lab in [(runs[0], BLUE, "All cells"), (runs[1], ORANGE, "NK cells removed")]:
    s = r["obs"]["score"] if "score" in r["obs"] else pd.Series(dtype=float)
    ax.hist(s, bins=bins, color=color, alpha=0.6, label=f"{lab} (n={len(s):,})")
if not len(runs[1]["obs"]):
    # Empty SIGMA output: no target component passed M, or no candidate
    passed = summary.loc[summary.run == "no_NK", "target_passed_M"].iloc[0]
    ax.text(0.5, 0.5, "NK cells removed:\nno candidate cell" if passed else
            "NK cells removed:\nno target component above M", transform=ax.transAxes,
            ha="center", va="center", fontsize=6.5, color=ORANGE)
ax.axvline(SCORE_THRESHOLD, color=INK, lw=0.8, ls=(0, (3, 2)))
ax.set_yscale("log")
ax.set_xlabel("SIGMA score")
ax.set_ylabel("Candidate cells (log scale)")
ax.set_title("Score distribution")
ax.legend(frameon=False, loc="upper center")
panel_label(ax, "A")

# B: selected cells, NK vs other author labels
ax = axes[1]
x = np.arange(len(summary))
ax.bar(x, summary["selected_nk"], color=BLUE, label="Author NK label", width=0.55)
ax.bar(x, summary["selected_other"], bottom=summary["selected_nk"], color=MUTED,
       label="Other label", width=0.55)
for xi, n in zip(x, summary["selected"]):
    ax.text(xi, n, f"{n:,}", ha="center", va="bottom", fontsize=6.5, color=INK2)
ax.set_xticks(x)
ax.set_xticklabels(["All cells", "NK cells removed"])
ax.set_ylabel("Cells with score ≥ 0.95")
ax.set_title("Selected cells")
ax.legend(frameon=False)
panel_label(ax, "B")

fig.savefig(os.path.join(FIG_DIR, "nk_absent.pdf"), bbox_inches="tight", dpi=500)
fig.savefig(os.path.join(FIG_DIR, "nk_absent.png"), bbox_inches="tight", dpi=200)
plt.show()